# Initial Setup

Library installation

Random seed initialization

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 35.8 MB/s eta 0:00:00


In [ ]:
!pip install -U transformers bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import random

seed = 9514 #random.randint(1000, 9999)

# Data Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_path = '/content/drive/My Drive/Apziva/Project 3 - Potential Talents/potential-talents.csv'

In [ ]:
!pip install duckdb

In [ ]:
import duckdb

In [ ]:
duckdb_conn = duckdb.connect(database=':memory:', read_only=False)

In [ ]:
data = duckdb_conn.sql(f"SELECT * FROM '{data_path}'")

# Data cleaning

In [ ]:
import nltk
import string

# Download the 'stopwords' corpus if not already downloaded
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords

# Get the list of English stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenize words and remove stopwords
    words = [word for word in text.split() if word not in stop_words]

    # Join the words back into a string
    return ' '.join(words)

In [ ]:
import duckdb # Keep the main duckdb import

try:
    # Register the Python function as a DuckDB UDF
    # Using 'str' as the return_type and for parameters instead of duckdb.typing.VARCHAR
    duckdb_conn.create_function("clean_text_udf", clean_text, return_type=str, parameters=[str])
except duckdb.NotImplementedException as e:
    if "A function by the name of 'clean_text_udf' is already created" in str(e):
        print("UDF 'clean_text_udf' already exists. Skipping creation.")
    else:
        raise # Re-raise other NotImplementedExceptions

# Apply the UDF to 'job_title' and 'location' columns and create a new DuckDB relation
data = duckdb_conn.sql("""
SELECT
    *, -- Select all existing columns
    clean_text_udf(job_title) AS cleaned_job_title,
    clean_text_udf(location) AS cleaned_location
FROM data
""")

print("Cleaning applied to 'job_title' and 'location' columns. The 'data' relation has been updated.")
print("New 'data' relation head with cleaned columns:")
data.show()

Cleaning applied to 'job_title' and 'location' columns. The 'data' relation has been updated.
New 'data' relation head with cleaned columns:
┌───────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────┬────────────┬─────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────┐
│  id   │                                                       job_title                                                       │              location               │ connection │   fit   │                                             cleaned_job_title                                             │         cleaned_location          │
│ int64 │                                                        varchar                                                        │               varchar               │  varchar   │ varchar 

# Tf-Idf Vectorization

I use Tf(Term Frequency)-Idf (Inverse Document Frequency) to convert job titles into numerical vectors, weighting words by their importance and enabling mathematical computations.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert DuckDB relation to Pandas DataFrame
df = data.fetchdf()

# Initialize TfidfVectorizer
vectorizer_job_title = TfidfVectorizer(stop_words='english', max_features=5000)

# Fill NaN values in 'job_title' column with empty strings to prevent errors during vectorization
df['cleaned_job_title'] = df['cleaned_job_title'].fillna('')

# Fit and transform the 'job_title' column
job_title_tfidf_matrix = vectorizer_job_title.fit_transform(df['cleaned_job_title'])

job_title_feature_names = vectorizer_job_title.get_feature_names_out()

print("TF-IDF matrix shape:", job_title_tfidf_matrix.shape)
print("Features (words):", vectorizer_job_title.get_feature_names_out()[:20]) # Print first 20 features

tf_idf_job_title_df = pd.DataFrame(job_title_tfidf_matrix.toarray(), columns=job_title_feature_names)
print(tf_idf_job_title_df)

TF-IDF matrix shape: (104, 178)
Features (words): ['2019' '2020' '408' '7092621' 'administration' 'administrative'
 'admissions' 'advisory' 'america' 'analyst' 'analytics' 'army' 'arts'
 'aspiring' 'assistant' 'atlanta' 'bachelor' 'bauer' 'bayar' 'beach']
         2019      2020  408  7092621  administration  administrative  \
0    0.321105  0.000000  0.0      0.0        0.000000             0.0   
1    0.000000  0.000000  0.0      0.0        0.000000             0.0   
2    0.000000  0.000000  0.0      0.0        0.000000             0.0   
3    0.000000  0.000000  0.0      0.0        0.000000             0.0   
4    0.000000  0.000000  0.0      0.0        0.000000             0.0   
..        ...       ...  ...      ...             ...             ...   
99   0.000000  0.364453  0.0      0.0        0.000000             0.0   
100  0.000000  0.000000  0.0      0.0        0.000000             0.0   
101  0.000000  0.000000  0.0      0.0        0.000000             0.0   
102  0.000000 

## Ranking Candidates Based on the fitness score from TF-Idf Vectorization

I have implemented a system that ranks the candidates based on the fitness with the given search keywords.

By representing both the job title dataset and search keywords in a shared TF-IDF vector space, the system computes the similiary by calculating the cosine similarity scores of the vector represenations.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack

class RankCandidatesTfIdf:
  def __init__(self, job_title_vectorizer, combined_features_matrix, dataframe):
    self.job_title_vectorizer = job_title_vectorizer
    self.combined_features_matrix = combined_features_matrix
    self.dataframe = dataframe

  def return_top_profiles(self, search_term: str, n_profiles: int):
    # Vectorize the search term for job title
    search_term_job_title_vector = self.job_title_vectorizer.transform([search_term])

    # Calculate cosine similarity between the search term and all combined profiles
    search_term_similarities = cosine_similarity(search_term_job_title_vector, self.combined_features_matrix).flatten()

    # Get the indices that would sort these similarities in descending order
    sorted_indices = search_term_similarities.argsort()[::-1]

    # Select the top n_profiles indices
    top_profile_indices = sorted_indices[:n_profiles]

    # Create a list of dictionaries for the top profiles
    ranked_profiles = []
    for idx in top_profile_indices:
        profile_info = {
            "id": self.dataframe.loc[idx, 'id'],
            "job_title": self.dataframe.loc[idx, 'job_title'],
            "location": self.dataframe.loc[idx, 'location'],
            "similarity_score": search_term_similarities[idx]
        }
        ranked_profiles.append(profile_info)

    return ranked_profiles


In [ ]:
rank_profile_obj = RankCandidatesTfIdf(vectorizer_job_title, job_title_tfidf_matrix, df)

In [ ]:
rank_profile_obj.return_top_profiles("Human Resources", 5)

[{'id': np.int64(74),
  'job_title': 'Human Resources Professional',
  'location': 'Greater Boston Area',
  'similarity_score': np.float64(0.6375192019339917)},
 {'id': np.int64(73),
  'job_title': 'Aspiring Human Resources Manager, seeking internship in Human Resources.',
  'location': 'Houston, Texas Area',
  'similarity_score': np.float64(0.547443027634779)},
 {'id': np.int64(97),
  'job_title': 'Aspiring Human Resources Professional',
  'location': 'Kokomo, Indiana Area',
  'similarity_score': np.float64(0.5439387582130529)},
 {'id': np.int64(21),
  'job_title': 'Aspiring Human Resources Professional',
  'location': 'Raleigh-Durham, North Carolina Area',
  'similarity_score': np.float64(0.5439387582130529)},
 {'id': np.int64(17),
  'job_title': 'Aspiring Human Resources Professional',
  'location': 'Raleigh-Durham, North Carolina Area',
  'similarity_score': np.float64(0.5439387582130529)}]

In [ ]:
rank_profile_obj.return_top_profiles("Aspiring human resources", 3)

[{'id': np.int64(97),
  'job_title': 'Aspiring Human Resources Professional',
  'location': 'Kokomo, Indiana Area',
  'similarity_score': np.float64(0.7535909978974563)},
 {'id': np.int64(33),
  'job_title': 'Aspiring Human Resources Professional',
  'location': 'Raleigh-Durham, North Carolina Area',
  'similarity_score': np.float64(0.7535909978974563)},
 {'id': np.int64(3),
  'job_title': 'Aspiring Human Resources Professional',
  'location': 'Raleigh-Durham, North Carolina Area',
  'similarity_score': np.float64(0.7535909978974563)}]

In [ ]:
rank_profile_obj.return_top_profiles("seeking human resources", 3)

[{'id': np.int64(30),
  'job_title': 'Seeking Human Resources Opportunities',
  'location': 'Chicago, Illinois',
  'similarity_score': np.float64(0.6649332283463288)},
 {'id': np.int64(28),
  'job_title': 'Seeking Human Resources Opportunities',
  'location': 'Chicago, Illinois',
  'similarity_score': np.float64(0.6649332283463288)},
 {'id': np.int64(99),
  'job_title': 'Seeking Human Resources Position',
  'location': 'Las Vegas, Nevada Area',
  'similarity_score': np.float64(0.6448509778309305)}]

In [ ]:
rank_profile_obj.return_top_profiles("Software", 3)

[{'id': np.int64(84),
  'job_title': 'Human Resources professional for the world leader in GIS software',
  'location': 'Highland, California',
  'similarity_score': np.float64(0.48171917384456087)},
 {'id': np.int64(104),
  'job_title': 'Director Of Administration at Excellence Logging',
  'location': 'Katy, Texas',
  'similarity_score': np.float64(0.0)},
 {'id': np.int64(102),
  'job_title': 'Business Intelligence and Analytics at Travelers',
  'location': 'Greater New York City Area',
  'similarity_score': np.float64(0.0)}]

In [ ]:
rank_profile_obj.return_top_profiles("Business", 3)

[{'id': np.int64(72),
  'job_title': 'Business Management Major and Aspiring Human Resources Manager',
  'location': 'Monroe, Louisiana Area',
  'similarity_score': np.float64(0.3969642939291679)},
 {'id': np.int64(102),
  'job_title': 'Business Intelligence and Analytics at Travelers',
  'location': 'Greater New York City Area',
  'similarity_score': np.float64(0.34603605269646276)},
 {'id': np.int64(81),
  'job_title': 'Senior Human Resources Business Partner at Heil Environmental',
  'location': 'Chattanooga, Tennessee Area',
  'similarity_score': np.float64(0.3133472484941707)}]

In [ ]:
rank_profile_obj.return_top_profiles("blah", 3)

[{'id': np.int64(104),
  'job_title': 'Director Of Administration at Excellence Logging',
  'location': 'Katy, Texas',
  'similarity_score': np.float64(0.0)},
 {'id': np.int64(103),
  'job_title': 'Always set them up for Success',
  'location': 'Greater Los Angeles Area',
  'similarity_score': np.float64(0.0)},
 {'id': np.int64(102),
  'job_title': 'Business Intelligence and Analytics at Travelers',
  'location': 'Greater New York City Area',
  'similarity_score': np.float64(0.0)}]

# Word Embedding

Tokenization: I tokenize the job title corpus and the search term by segmenting into individual words, treating each word as a distinct token.

## word2vec

word2vec is superior to tf-idf, but it has some shortcomings. For example, the fruit 'apple' and the Mac 'apple' are mapped to the same embeddings.

In [ ]:
from gensim.models import Word2Vec

# Tokenize the cleaned job titles into lists of words
tokenized_job_titles = [title.split() for title in df['cleaned_job_title']]

# Train the Word2Vec model
# window: maximum distance between the current and predicted word within a sentence
# min_count: Ignores all words with total frequency lower than this
# workers: Use these many worker threads to train the model
# seed: Seed for the random number generator, for reproducibility
word2vec_model = Word2Vec(sentences=tokenized_job_titles, vector_size=100, window=5, min_count=1, workers=1, seed=seed)

In [ ]:
word2vec_model.wv["human"]

array([-5.9152381e-03, -2.6968075e-03, -1.5343941e-03, -5.0976002e-03,
       -6.9142012e-03,  6.0090707e-03,  5.1979902e-03,  9.0153534e-03,
        8.0017000e-03, -7.5834668e-03, -5.9088003e-03,  2.7579004e-03,
        8.6597512e-03, -8.1006018e-03, -6.5313699e-03,  9.8591612e-04,
       -9.0751229e-03,  2.5883680e-03, -2.0894301e-03, -1.6212407e-03,
       -6.0914136e-03,  3.5206235e-03, -3.0131908e-03,  2.0821884e-03,
        3.6356868e-03, -6.7392229e-03,  1.9735850e-03,  5.8846958e-03,
        5.8228704e-03, -4.7258058e-04,  7.3038586e-03, -3.1477665e-03,
       -2.8815919e-03, -6.8486365e-04,  3.0971970e-04, -8.9085801e-03,
       -4.6188328e-03, -3.3424911e-03, -3.7590687e-03, -2.9012668e-03,
       -7.1161641e-03,  6.2174466e-03, -5.7525379e-03,  6.6746944e-03,
       -9.7030587e-03,  5.7526249e-03,  7.1490468e-03,  6.3909744e-03,
       -3.3946845e-03,  6.2694098e-03, -2.4358279e-03,  9.6385684e-03,
       -7.1207215e-03, -4.1405484e-03, -7.3408177e-03,  2.8443804e-03,
      

In [ ]:
word2vec_model.wv["AI"]

KeyError: "Key 'AI' not present"

Limitation of wordvec: I get a KeyError if I try to find the word embedding of a word that's not in the training data.

### Ranking Candidates Based on the similarity score from word2vec embedding

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class RankCandidatesWord2vec:
    def __init__(self, word2vec_model, dataframe):
        self.word2vec_model = word2vec_model
        self.dataframe = dataframe
        self.vector_size = self.word2vec_model.vector_size

    def _get_phrase_vector(self, phrase_words):
        # Filter out words not in the model's vocabulary
        valid_words = [word for word in phrase_words if word in self.word2vec_model.wv.key_to_index]

        if not valid_words:
            # If no words are in vocabulary, return a zero vector
            return np.zeros((1, self.vector_size))

        # Get vectors for valid words and calculate their mean
        vectors = [self.word2vec_model.wv[word] for word in valid_words]
        return np.mean(vectors, axis=0).reshape(1, -1)

    def return_top_profiles(self, search_term: str, n_profiles: int):
        # Clean and tokenize the search term
        cleaned_search_term = clean_text(search_term)
        tokenized_search_term = cleaned_search_term.split()

        # Get the vector for the search term
        search_term_vector = self._get_phrase_vector(tokenized_search_term)

        # Get vectors for all cleaned job titles in the dataframe
        # Ensure 'cleaned_job_title' is treated as a list of words for tokenization
        job_title_vectors = np.vstack([
            self._get_phrase_vector(str(title).split())
            for title in self.dataframe['cleaned_job_title'].tolist()
        ])

        # Calculate cosine similarity between the search term vector and all job title vectors
        similarities = cosine_similarity(search_term_vector, job_title_vectors).flatten()

        # Get the indices that would sort these similarities in descending order
        sorted_indices = similarities.argsort()[::-1]

        # Select the top n_profiles indices
        top_profile_indices = sorted_indices[:n_profiles]

        # Create a list of dictionaries for the top profiles
        ranked_profiles = []
        for idx in top_profile_indices:
            profile_info = {
                "id": self.dataframe.loc[idx, 'id'],
                "job_title": self.dataframe.loc[idx, 'job_title'],
                "location": self.dataframe.loc[idx, 'location'],
                "similarity_score": similarities[idx]
            }
            ranked_profiles.append(profile_info)

        return ranked_profiles

In [ ]:
rank_profile_word_embedding_obj = RankCandidatesWord2vec(word2vec_model, df)

In [ ]:
rank_profile_word_embedding_obj.return_top_profiles("Human Resources", 5)

[{'id': np.int64(74),
  'job_title': 'Human Resources Professional',
  'location': 'Greater Boston Area',
  'similarity_score': np.float32(0.8579174)},
 {'id': np.int64(73),
  'job_title': 'Aspiring Human Resources Manager, seeking internship in Human Resources.',
  'location': 'Houston, Texas Area',
  'similarity_score': np.float32(0.839593)},
 {'id': np.int64(97),
  'job_title': 'Aspiring Human Resources Professional',
  'location': 'Kokomo, Indiana Area',
  'similarity_score': np.float32(0.79859436)},
 {'id': np.int64(21),
  'job_title': 'Aspiring Human Resources Professional',
  'location': 'Raleigh-Durham, North Carolina Area',
  'similarity_score': np.float32(0.79859436)},
 {'id': np.int64(17),
  'job_title': 'Aspiring Human Resources Professional',
  'location': 'Raleigh-Durham, North Carolina Area',
  'similarity_score': np.float32(0.79859436)}]

In [ ]:
rank_profile_word_embedding_obj.return_top_profiles("Aspiring human resources", 3)

[{'id': np.int64(24),
  'job_title': 'Aspiring Human Resources Specialist',
  'location': 'Greater New York City Area',
  'similarity_score': np.float32(0.89259577)},
 {'id': np.int64(36),
  'job_title': 'Aspiring Human Resources Specialist',
  'location': 'Greater New York City Area',
  'similarity_score': np.float32(0.89259577)},
 {'id': np.int64(49),
  'job_title': 'Aspiring Human Resources Specialist',
  'location': 'Greater New York City Area',
  'similarity_score': np.float32(0.89259577)}]

In [ ]:
rank_profile_word_embedding_obj.return_top_profiles("Software", 3)

[{'id': np.int64(84),
  'job_title': 'Human Resources professional for the world leader in GIS software',
  'location': 'Highland, California',
  'similarity_score': np.float32(0.3278067)},
 {'id': np.int64(34),
  'job_title': 'People Development Coordinator at Ryan',
  'location': 'Denton, Texas',
  'similarity_score': np.float32(0.17579666)},
 {'id': np.int64(4),
  'job_title': 'People Development Coordinator at Ryan',
  'location': 'Denton, Texas',
  'similarity_score': np.float32(0.17579666)}]

In [ ]:
rank_profile_word_embedding_obj.return_top_profiles("Data", 3)

[{'id': np.int64(86),
  'job_title': 'Information Systems Specialist and Programmer with a love for data and organization.',
  'location': 'Gaithersburg, Maryland',
  'similarity_score': np.float32(0.44628224)},
 {'id': np.int64(72),
  'job_title': 'Business Management Major and Aspiring Human Resources Manager',
  'location': 'Monroe, Louisiana Area',
  'similarity_score': np.float32(0.14968173)},
 {'id': np.int64(76),
  'job_title': 'Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment',
  'location': 'New York, New York',
  'similarity_score': np.float32(0.13735649)}]

## GloVe: Global Vectors

In [ ]:
import os

# Define the URL for the GloVe embeddings
glove_url = "http://nlp.stanford.edu/data/glove.6B.zip"
zip_file_name = "glove.6B.zip"

# Download the GloVe embeddings if not already downloaded
if not os.path.exists(zip_file_name):
    print(f"Downloading {zip_file_name}...")
    !wget {glove_url}
else:
    print(f"{zip_file_name} already exists. Skipping download.")

# Extract the GloVe embeddings if not already extracted (check for one of the expected files)
expected_glove_file = "glove.6B.100d.txt"
if not os.path.exists(expected_glove_file):
    print(f"Extracting {zip_file_name}...")
    !unzip -q {zip_file_name}
else:
    print(f"GloVe embeddings (e.g., {expected_glove_file}) already extracted. Skipping extraction.")

print("GloVe download and extraction complete.")

--2026-03-27 12:55:50--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-03-27 12:55:50--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-03-27 12:55:50--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [ ]:
glove_model_path = "glove.6B.100d.txt"
glove_embeddings = {}

with open(glove_model_path, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        glove_embeddings[word] = vector

In [ ]:
glove_embeddings['human'][:10]

array([ 0.33864  ,  0.59663  ,  0.53322  ,  0.31404  ,  0.15321  ,
        0.31749  , -0.4294   , -0.2915   , -0.0021047, -0.39309  ],
      dtype=float32)

In [ ]:
glove_embeddings['AI'][:10]

KeyError: 'AI'

Limitation of GloVe: I get a KeyError if I try to find the word embedding of a word that's not in the training data.

### Ranking Candidates Based on the similarity score from GloVe Embedding

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class RankCandidatesGloVe:
    def __init__(self, glove_embeddings, dataframe):
        self.glove_embeddings = glove_embeddings
        self.dataframe = dataframe
        # Assuming 100d embeddings as per `glove.6B.100d.txt`
        self.vector_size = 100

    def _get_phrase_vector(self, phrase_words):
        # Filter out words not in the model's vocabulary
        valid_words = [word for word in phrase_words if word in self.glove_embeddings]

        if not valid_words:
            # If no words are in vocabulary, return a zero vector
            return np.zeros((1, self.vector_size))

        # Get vectors for valid words and calculate their mean
        vectors = [self.glove_embeddings[word] for word in valid_words]
        return np.mean(vectors, axis=0).reshape(1, -1)

    def return_top_profiles(self, search_term: str, n_profiles: int):
        # Clean and tokenize the search term using the existing clean_text function
        cleaned_search_term = clean_text(search_term)
        tokenized_search_term = cleaned_search_term.split()

        # Get the vector for the search term
        search_term_vector = self._get_phrase_vector(tokenized_search_term)

        # Get vectors for all cleaned job titles in the dataframe
        job_title_vectors = np.vstack([
            self._get_phrase_vector(str(title).split())
            for title in self.dataframe['cleaned_job_title'].tolist()
        ])

        # Calculate cosine similarity between the search term vector and all job title vectors
        similarities = cosine_similarity(search_term_vector, job_title_vectors).flatten()

        # Get the indices that would sort these similarities in descending order
        sorted_indices = similarities.argsort()[::-1]

        # Select the top n_profiles indices
        top_profile_indices = sorted_indices[:n_profiles]

        # Create a list of dictionaries for the top profiles
        ranked_profiles = []
        for idx in top_profile_indices:
            profile_info = {
                "id": self.dataframe.loc[idx, 'id'],
                "job_title": self.dataframe.loc[idx, 'job_title'],
                "location": self.dataframe.loc[idx, 'location'],
                "similarity_score": similarities[idx]
            }
            ranked_profiles.append(profile_info)

        return ranked_profiles

In [ ]:
rank_profile_glove_obj = RankCandidatesGloVe(glove_embeddings, df)

In [ ]:
rank_profile_glove_obj.return_top_profiles("Human Resources", 5)

[{'id': np.int64(101),
  'job_title': 'Human Resources Generalist at Loparex',
  'location': 'Raleigh-Durham, North Carolina Area',
  'similarity_score': np.float32(0.9271207)},
 {'id': np.int64(78),
  'job_title': "Human Resources Generalist at Schwan's",
  'location': 'Amerika Birleşik Devletleri',
  'similarity_score': np.float32(0.9271207)},
 {'id': np.int64(74),
  'job_title': 'Human Resources Professional',
  'location': 'Greater Boston Area',
  'similarity_score': np.float32(0.9202033)},
 {'id': np.int64(28),
  'job_title': 'Seeking Human Resources Opportunities',
  'location': 'Chicago, Illinois',
  'similarity_score': np.float32(0.9121238)},
 {'id': np.int64(30),
  'job_title': 'Seeking Human Resources Opportunities',
  'location': 'Chicago, Illinois',
  'similarity_score': np.float32(0.9121238)}]

In [ ]:
rank_profile_glove_obj.return_top_profiles("AI", 3)

[{'id': np.int64(103),
  'job_title': 'Always set them up for Success',
  'location': 'Greater Los Angeles Area',
  'similarity_score': np.float32(0.3870576)},
 {'id': np.int64(86),
  'job_title': 'Information Systems Specialist and Programmer with a love for data and organization.',
  'location': 'Gaithersburg, Maryland',
  'similarity_score': np.float32(0.33819702)},
 {'id': np.int64(84),
  'job_title': 'Human Resources professional for the world leader in GIS software',
  'location': 'Highland, California',
  'similarity_score': np.float32(0.30733305)}]

## FastText

- character n-gram
  - resilient to typos
  - supports morphologically rich languages
  - byte pair encoding
    - improved computational efficiency
    - "based" and "base" have similar semantic meanings: encodes "base" and "d"
    - "encoding": "encod" and "ing"

In [ ]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.2-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.2-py3-none-any.whl (310 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4647420 sha256=ea3823746c613a52a9fe7b8bfc75a55f1837ba379e90f476915c9844557743f6
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
fasttext_input_file = "cleaned_job_titles.txt"
with open(fasttext_input_file, 'w') as f:
    for item in df['cleaned_job_title'].dropna():
        f.write(item + '\n')

In [ ]:
import fasttext

In [ ]:
fasttext_model = fasttext.train_unsupervised(fasttext_input_file, model='skipgram', dim=100, epoch=10, minCount=1)

In [ ]:
print(fasttext_model.get_word_vector("human")[:10])

[ 4.4122807e-04 -4.7833481e-04 -9.1423477e-05 -2.4253734e-04
  2.3381428e-05 -4.6130156e-04 -5.5204681e-04 -6.2492763e-04
  6.9250591e-04  6.6774746e-04]


In [ ]:
print(fasttext_model.get_word_vector("AI")[:10])

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


### Ranking Candidates Based on the similarity score from fasttext

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class RankCandidatesFastText:
    def __init__(self, fasttext_model, dataframe):
        self.fasttext_model = fasttext_model
        self.dataframe = dataframe
        # FastText model dimension is set to 100 during training (dim=100)
        self.vector_size = self.fasttext_model.get_dimension()

    def _get_phrase_vector(self, phrase_words):
        # Filter out words not in the model's vocabulary (though FastText handles OOV words)
        # For simplicity, we'll still check for words that might not produce meaningful vectors
        # or average only from words that have actual vectors.

        # FastText can generate vectors for OOV words by using subword information,
        # so we don't strictly need to check if word in model.words.

        vectors = [self.fasttext_model.get_word_vector(word) for word in phrase_words if word.strip() != '']

        if not vectors:
            # If no words in the phrase, return a zero vector
            return np.zeros((1, self.vector_size))

        # Average the word vectors
        return np.mean(vectors, axis=0).reshape(1, -1)

    def return_top_profiles(self, search_term: str, n_profiles: int):
        # Clean and tokenize the search term using the existing clean_text function
        cleaned_search_term = clean_text(search_term)
        tokenized_search_term = cleaned_search_term.split()

        # Get the vector for the search term
        search_term_vector = self._get_phrase_vector(tokenized_search_term)

        # Initialize a list to hold job title vectors
        job_title_vectors_list = []
        for title in self.dataframe['cleaned_job_title'].tolist():
            # Ensure the title is treated as a string before splitting
            tokenized_title = str(title).split()
            job_title_vectors_list.append(self._get_phrase_vector(tokenized_title))

        # Stack the list of vectors into a NumPy array
        job_title_vectors = np.vstack(job_title_vectors_list)

        # Calculate cosine similarity between the search term vector and all job title vectors
        similarities = cosine_similarity(search_term_vector, job_title_vectors).flatten()

        # Get the indices that would sort these similarities in descending order
        sorted_indices = similarities.argsort()[::-1]

        # Select the top n_profiles indices
        top_profile_indices = sorted_indices[:n_profiles]

        # Create a list of dictionaries for the top profiles
        ranked_profiles = []
        for idx in top_profile_indices:
            profile_info = {
                "id": self.dataframe.loc[idx, 'id'],
                "job_title": self.dataframe.loc[idx, 'job_title'],
                "location": self.dataframe.loc[idx, 'location'],
                "similarity_score": similarities[idx]
            }
            ranked_profiles.append(profile_info)

        return ranked_profiles

In [ ]:
rank_profile_fasttext_obj = RankCandidatesFastText(fasttext_model, df)

In [ ]:
rank_profile_fasttext_obj.return_top_profiles("Human Resources", 5)

[{'id': np.int64(74),
  'job_title': 'Human Resources Professional',
  'location': 'Greater Boston Area',
  'similarity_score': np.float32(0.9216496)},
 {'id': np.int64(73),
  'job_title': 'Aspiring Human Resources Manager, seeking internship in Human Resources.',
  'location': 'Houston, Texas Area',
  'similarity_score': np.float32(0.8066384)},
 {'id': np.int64(101),
  'job_title': 'Human Resources Generalist at Loparex',
  'location': 'Raleigh-Durham, North Carolina Area',
  'similarity_score': np.float32(0.78805536)},
 {'id': np.int64(97),
  'job_title': 'Aspiring Human Resources Professional',
  'location': 'Kokomo, Indiana Area',
  'similarity_score': np.float32(0.7448012)},
 {'id': np.int64(17),
  'job_title': 'Aspiring Human Resources Professional',
  'location': 'Raleigh-Durham, North Carolina Area',
  'similarity_score': np.float32(0.7448012)}]

In [ ]:
rank_profile_fasttext_obj.return_top_profiles("Software", 3)

[{'id': np.int64(84),
  'job_title': 'Human Resources professional for the world leader in GIS software',
  'location': 'Highland, California',
  'similarity_score': np.float32(0.3352996)},
 {'id': np.int64(93),
  'job_title': 'Admissions Representative at Community medical center long beach',
  'location': 'Long Beach, California',
  'similarity_score': np.float32(0.2580709)},
 {'id': np.int64(71),
  'job_title': 'Human Resources Generalist at ScottMadden, Inc.',
  'location': 'Raleigh-Durham, North Carolina Area',
  'similarity_score': np.float32(0.22797754)}]

In [ ]:
rank_profile_fasttext_obj.return_top_profiles("AI", 3)

[{'id': np.int64(104),
  'job_title': 'Director Of Administration at Excellence Logging',
  'location': 'Katy, Texas',
  'similarity_score': np.float32(0.0)},
 {'id': np.int64(103),
  'job_title': 'Always set them up for Success',
  'location': 'Greater Los Angeles Area',
  'similarity_score': np.float32(0.0)},
 {'id': np.int64(102),
  'job_title': 'Business Intelligence and Analytics at Travelers',
  'location': 'Greater New York City Area',
  'similarity_score': np.float32(0.0)}]

# Contextualized Embedding


In [ ]:
!pip install tensorflow
!pip install tensorflow-hub

## ELMo

In [ ]:
elmo_model = hub.load("https://tfhub.dev/google/elmo/3")

def get_elmo_embedding(text_list):
    if not text_list or all(not s.strip() for s in text_list):
        # Return a zero vector if the input list is empty or contains only empty strings
        # ELMo default output dimension is 1024 for the 'default' signature.
        return np.zeros(1024)

    # The ELMo module expects a list of strings for signature='default'
    # It returns a dictionary with 'elmo', 'default', 'word_emb' keys.
    # 'default' is typically the average of the 3 layers.
    embeddings = elmo_model.signatures["default"](tf.constant(text_list))["default"]
    return embeddings.numpy().mean(axis=0)

# Apply ELMo embeddings to the 'cleaned_job_title' column
# This will be computationally intensive and might take a long time.
# Ensure 'cleaned_job_title' column has string values for ELMo.
df['elmo_embedding'] = df['cleaned_job_title'].apply(lambda x: get_elmo_embedding([str(x)]))

print("ELMo embeddings generated.")
print(f"Shape of first ELMo embedding: {df['elmo_embedding'].iloc[0].shape}")

class RankCandidatesELMo:
    def __init__(self, elmo_model, dataframe, embedding_column='elmo_embedding'):
        self.elmo_model = elmo_model
        self.dataframe = dataframe
        self.embedding_column = embedding_column
        self.vector_size = 1024 # ELMo's 'default' output dimension

    def _get_search_term_vector(self, search_term: str):
        if not search_term:
            return np.zeros(self.vector_size).reshape(1, -1)

        cleaned_search_term = clean_text(search_term)
        return get_elmo_embedding([cleaned_search_term]).reshape(1, -1)

    def return_top_profiles(self, search_term: str, n_profiles: int):
        search_term_vector = self._get_search_term_vector(search_term)

        # Stack the ELMo embeddings from the DataFrame into a single NumPy array
        job_title_vectors = np.vstack(self.dataframe[self.embedding_column].tolist())

        similarities = cosine_similarity(search_term_vector, job_title_vectors).flatten()

        sorted_indices = similarities.argsort()[::-1]
        top_profile_indices = sorted_indices[:n_profiles]

        ranked_profiles = []
        for idx in top_profile_indices:
            profile_info = {
                "id": self.dataframe.loc[idx, 'id'],
                "job_title": self.dataframe.loc[idx, 'job_title'],
                "location": self.dataframe.loc[idx, 'location'],
                "similarity_score": similarities[idx]
            }
            ranked_profiles.append(profile_info)

        return ranked_profiles

ELMo embeddings generated.
Shape of first ELMo embedding: (1024,)
[{'id': np.int64(74), 'job_title': 'Human Resources Professional', 'location': 'Greater Boston Area', 'similarity_score': np.float32(0.8433589)}, {'id': np.int64(78), 'job_title': "Human Resources Generalist at Schwan's", 'location': 'Amerika Birleşik Devletleri', 'similarity_score': np.float32(0.7674339)}, {'id': np.int64(101), 'job_title': 'Human Resources Generalist at Loparex', 'location': 'Raleigh-Durham, North Carolina Area', 'similarity_score': np.float32(0.76362324)}, {'id': np.int64(88), 'job_title': 'Human Resources Management Major', 'location': 'Milpitas, California', 'similarity_score': np.float32(0.72693616)}, {'id': np.int64(73), 'job_title': 'Aspiring Human Resources Manager, seeking internship in Human Resources.', 'location': 'Houston, Texas Area', 'similarity_score': np.float32(0.7257056)}]
[{'id': np.int64(97), 'job_title': 'Aspiring Human Resources Professional', 'location': 'Kokomo, Indiana Area', '

In [ ]:
rank_profile_elmo_obj = RankCandidatesELMo(elmo_model, df)

In [ ]:
print(rank_profile_elmo_obj.return_top_profiles("Human Resources", 5))

[{'id': np.int64(74), 'job_title': 'Human Resources Professional', 'location': 'Greater Boston Area', 'similarity_score': np.float32(0.8433589)}, {'id': np.int64(78), 'job_title': "Human Resources Generalist at Schwan's", 'location': 'Amerika Birleşik Devletleri', 'similarity_score': np.float32(0.7674339)}, {'id': np.int64(101), 'job_title': 'Human Resources Generalist at Loparex', 'location': 'Raleigh-Durham, North Carolina Area', 'similarity_score': np.float32(0.76362324)}, {'id': np.int64(88), 'job_title': 'Human Resources Management Major', 'location': 'Milpitas, California', 'similarity_score': np.float32(0.72693616)}, {'id': np.int64(73), 'job_title': 'Aspiring Human Resources Manager, seeking internship in Human Resources.', 'location': 'Houston, Texas Area', 'similarity_score': np.float32(0.7257056)}]


In [ ]:
print(rank_profile_elmo_obj.return_top_profiles("Aspiring human resources", 3))
print(rank_profile_elmo_obj.return_top_profiles("Software", 3))
print(rank_profile_elmo_obj.return_top_profiles("Data", 3))
print(rank_profile_elmo_obj.return_top_profiles("AI", 3))

[{'id': np.int64(97), 'job_title': 'Aspiring Human Resources Professional', 'location': 'Kokomo, Indiana Area', 'similarity_score': np.float32(0.87471354)}, {'id': np.int64(33), 'job_title': 'Aspiring Human Resources Professional', 'location': 'Raleigh-Durham, North Carolina Area', 'similarity_score': np.float32(0.87471354)}, {'id': np.int64(3), 'job_title': 'Aspiring Human Resources Professional', 'location': 'Raleigh-Durham, North Carolina Area', 'similarity_score': np.float32(0.87471354)}]
[{'id': np.int64(84), 'job_title': 'Human Resources professional for the world leader in GIS software', 'location': 'Highland, California', 'similarity_score': np.float32(0.54951435)}, {'id': np.int64(86), 'job_title': 'Information Systems Specialist and Programmer with a love for data and organization.', 'location': 'Gaithersburg, Maryland', 'similarity_score': np.float32(0.5426526)}, {'id': np.int64(102), 'job_title': 'Business Intelligence and Analytics at Travelers', 'location': 'Greater New Y

## BERT

In [ ]:
!pip install transformers[sentencepiece] torch

import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import tensorflow_hub as hub
import tensorflow as tf

# Load pre-trained BERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

def get_bert_embedding(text):
    if not text:
        return torch.zeros(model.config.hidden_size)
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use the mean of the last hidden state as the sentence embedding
    return outputs.last_hidden_state.mean(dim=1).squeeze()

# Apply to job titles
# This might take a while depending on the size of your dataframe
df['bert_embedding'] = df['cleaned_job_title'].apply(get_bert_embedding)

print("BERT embeddings generated.")
print(f"Shape of first BERT embedding: {df['bert_embedding'].iloc[0].shape}")

class RankCandidatesBERT:
    def __init__(self, model, tokenizer, dataframe, embedding_column='bert_embedding'):
        self.model = model
        self.tokenizer = tokenizer
        self.dataframe = dataframe
        self.embedding_column = embedding_column
        self.vector_size = self.model.config.hidden_size

    def _get_search_term_vector(self, search_term: str):
        if not search_term:
            return torch.zeros(self.vector_size).unsqueeze(0)

        # Clean the search term using the existing clean_text function
        cleaned_search_term = clean_text(search_term)

        inputs = self.tokenizer(cleaned_search_term, return_tensors='pt', padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = self.model(**inputs)
        return outputs.last_hidden_state.mean(dim=1)

    def return_top_profiles(self, search_term: str, n_profiles: int):
        search_term_vector = self._get_search_term_vector(search_term).numpy()

        # Convert list of tensors to a single NumPy array for similarity calculation
        job_title_vectors = np.vstack([embedding.numpy() for embedding in self.dataframe[self.embedding_column]])

        similarities = cosine_similarity(search_term_vector, job_title_vectors).flatten()

        sorted_indices = similarities.argsort()[::-1]
        top_profile_indices = sorted_indices[:n_profiles]

        ranked_profiles = []
        for idx in top_profile_indices:
            profile_info = {
                "id": self.dataframe.loc[idx, 'id'],
                "job_title": self.dataframe.loc[idx, 'job_title'],
                "location": self.dataframe.loc[idx, 'location'],
                "similarity_score": similarities[idx]
            }
            ranked_profiles.append(profile_info)

        return ranked_profiles


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT embeddings generated.
Shape of first BERT embedding: torch.Size([768])


In [ ]:
rank_profile_bert_obj = RankCandidatesBERT(model, tokenizer, df)

print(rank_profile_bert_obj.return_top_profiles("Human Resources", 5))
print(rank_profile_bert_obj.return_top_profiles("Aspiring human resources", 3))
print(rank_profile_bert_obj.return_top_profiles("Software", 3))
print(rank_profile_bert_obj.return_top_profiles("Data", 3))
print(rank_profile_bert_obj.return_top_profiles("AI", 3))

[{'id': np.int64(74), 'job_title': 'Human Resources Professional', 'location': 'Greater Boston Area', 'similarity_score': np.float32(0.8730361)}, {'id': np.int64(88), 'job_title': 'Human Resources Management Major', 'location': 'Milpitas, California', 'similarity_score': np.float32(0.7823341)}, {'id': np.int64(89), 'job_title': 'Director Human Resources  at EY', 'location': 'Greater Atlanta Area', 'similarity_score': np.float32(0.77355504)}, {'id': np.int64(28), 'job_title': 'Seeking Human Resources Opportunities', 'location': 'Chicago, Illinois', 'similarity_score': np.float32(0.7523645)}, {'id': np.int64(30), 'job_title': 'Seeking Human Resources Opportunities', 'location': 'Chicago, Illinois', 'similarity_score': np.float32(0.7523645)}]
[{'id': np.int64(24), 'job_title': 'Aspiring Human Resources Specialist', 'location': 'Greater New York City Area', 'similarity_score': np.float32(0.90548)}, {'id': np.int64(36), 'job_title': 'Aspiring Human Resources Specialist', 'location': 'Greate

# LLM

In this section, I use LLMs to compute the similarity scores between job profiles and search terms, and then rank the candidates based on the scores. Using LLM feels like I am communicating with a human: it gives conversational insights and makes the results more interpretible.

There are three main types of LLM's:
- Chat based LLM

- Instruction based LLM

- Reasoning LLM model

I have opted for an instruction-based LLM. Since the client is requesting a functional ranking system rather than a chatbot, this model type is a better fit for this project.

LLM usage
- local model: constraints with the hardware resources (e.g. RAM, disk space), inference latency
- cloud-based API approach: billed based on token usage, low network latency

## Qwen model

### Atomic approach

In this approach, I ran the LLM on each individual data entry to compute the semantic similarity score between the search term and each candidate's job title. Using the returned computation results, I returned the top candidates.

#### Hugging Face

I import the LLM from the Hugging Face model hub.

In [ ]:
model_path = "Qwen/Qwen3.5-9B-Base"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", quantization_config=quantization_config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

In [ ]:
import torch
import re
import numpy as np
import duckdb # Import duckdb

# model and tokenizer are already loaded in previous cells.

class RankCandidatesLLM:
    def __init__(self, model, tokenizer, duckdb_relation):
        self.model = model
        self.tokenizer = tokenizer
        self.duckdb_relation = duckdb_relation # Store the DuckDB relation

    def _get_llm_response(self, prompt: str) -> str:
        # Encode the prompt
        inputs = self.tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(self.model.device)

        # Generate a response
        # Using max_new_tokens to control output length, adjust as needed.
        # do_sample=True for more varied responses, False for more deterministic.
        # temperature can control creativity.
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=100, # Limit the output length for efficiency and focused response
                do_sample=True,
                top_p=0.9,
                temperature=0.7,
                num_return_sequences=1,
                eos_token_id=self.tokenizer.eos_token_id # Stop generation at EOS token
            )

        # Decode the generated tokens
        # We need to slice the output to remove the input prompt tokens
        response = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        return response.strip()

    def _parse_llm_output(self, llm_output: str):
        score_match = re.search(r"Score:\s*(\d+\.?\d*)", llm_output)
        explanation_match = re.search(r"Explanation:\s*(.*)", llm_output, re.DOTALL)

        score = float(score_match.group(1)) if score_match else 0.0
        explanation = explanation_match.group(1).strip() if explanation_match else "No explanation provided."

        return score, explanation

    def return_top_profiles(self, search_term: str, n_profiles: int):
        ranked_profiles = []
        # Fetch data once from DuckDB relation into a Pandas DataFrame for easier iteration
        candidates_df = self.duckdb_relation.fetchdf()
        print(f"Processing {len(candidates_df)} candidates for search term '{search_term}'...")

        for index, row in candidates_df.iterrows():
            print(index)
            job_title = row['job_title']
            location = row['location']
            original_id = row['id'] # Assuming 'id' is the unique identifier

            # Construct the prompt for the LLM
            prompt = f"""Given the search term: '{search_term}' and a candidate's job title: '{job_title}'.
Rate the semantic similarity between the search term and the job title on a scale of 0.0 to 1.0, where 1.0 is a perfect match and 0.0 is no similarity.
Provide a brief explanation for your rating.
Format your response as:
Score: [float_score]
Explanation: [your_explanation]"""

            llm_response = self._get_llm_response(prompt)
            score, explanation = self._parse_llm_output(llm_response)

            profile_info = {
                "id": original_id,
                "job_title": job_title,
                "location": location,
                "similarity_score": score,
                "explanation": explanation
            }
            ranked_profiles.append(profile_info)

        # Sort profiles by similarity score in descending order
        ranked_profiles.sort(key=lambda x: x['similarity_score'], reverse=True)

        return ranked_profiles[:n_profiles]

In [ ]:
rank_profile_llm_obj = RankCandidatesLLM(model, tokenizer, data)

In [ ]:
print(rank_profile_llm_obj.return_top_profiles("Human Resources", 5))

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Processing 104 candidates for search term 'Human Resources'...
0


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


1


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


...

101


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


102


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


103
[{'id': 1, 'job_title': '2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional', 'location': 'Houston, Texas', 'similarity_score': 0.95, 'explanation': 'The job title explicitly identifies the candidate as an "aspiring Human Resources professional." This directly matches the search term, indicating a strong semantic alignment and relevance to the target role.'}, {'id': 13, 'job_title': 'Human Resources Coordinator at InterContinental Buckhead Atlanta', 'location': 'Atlanta, Georgia', 'similarity_score': 0.95, 'explanation': 'The job title is an exact match for the search term. The title "Human Resources Coordinator" explicitly includes the core keywords "Human Resources," indicating a perfect semantic alignment.'}, {'id': 15, 'job_title': '2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional', 'location': 'Houston, Texas', 'similarity_score': 0.95, 'explanation': 'The job title ex

{'id': 1, 'job_title': '2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional', 'location': 'Houston, Texas', 'similarity_score': 0.95, 'explanation': 'The job title explicitly identifies the candidate as an "aspiring Human Resources professional." This directly matches the search term, indicating a strong semantic alignment and relevance to the target role.'}

{'id': 13, 'job_title': 'Human Resources Coordinator at InterContinental Buckhead Atlanta', 'location': 'Atlanta, Georgia', 'similarity_score': 0.95, 'explanation': 'The job title is an exact match for the search term. The title "Human Resources Coordinator" explicitly includes the core keywords "Human Resources," indicating a perfect semantic alignment.'}

{'id': 15, 'job_title': '2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional', 'location': 'Houston, Texas', 'similarity_score': 0.95, 'explanation': 'The job title explicitly states the candidate is an "aspiring Human Resources professional." This is a direct match to the search term. The inclusion of the specific degree and year provides high contextual relevance, confirming the candidate\'s educational background in the field.'}

{'id': 27, 'job_title': 'Aspiring Human Resources Management student seeking an internship', 'location': 'Houston, Texas Area', 'similarity_score': 0.95, 'explanation': 'The job title explicitly states the field of study is "Human Resources Management," which directly matches the search term "Human Resources." The slight difference is the addition of "Aspiring" and "student seeking an internship," which are contextual modifiers rather than semantic differences in the core subject matter.'}

{'id': 28, 'job_title': 'Seeking Human Resources Opportunities', 'location': 'Chicago, Illinois', 'similarity_score': 0.95, 'explanation': "The search term 'Human Resources' is a direct match to the job title 'Seeking Human Resources Opportunities'. The term 'Seeking' indicates the candidate's intent to find a job in this specific field, and 'Opportunities' is a common synonym for job openings. Therefore, the two terms are semantically very similar, with a score of 0.95 indicating a strong match."}

In [ ]:
print(rank_profile_llm_obj.return_top_profiles("Software Engineer", 3))

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Processing 104 candidates for search term 'Software Engineer'...
0


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


1


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


...

101


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


102


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


103
[{'id': 80, 'job_title': 'Junior MES Engineer| Information Systems', 'location': 'Myrtle Beach, South Carolina Area', 'similarity_score': 0.75, 'explanation': 'The term "Software Engineer" shares significant semantic overlap with "Information Systems" due to the common domain of software development and IT infrastructure. However, the specific inclusion of "MES" (Manufacturing Execution System) in the candidate\'s title indicates a specialized focus on industrial automation and manufacturing software, which is a narrower domain than general software engineering. Consequently, while they are closely related, they are not a perfect match.'}, {'id': 34, 'job_title': 'People Development Coordinator at Ryan', 'location': 'Denton, Texas', 'similarity_score': 0.2, 'explanation': 'The search term "Software Engineer" requires technical proficiency in coding, software development, and computer science. The candidate\'s title, "People Development Coordinator," focuses on human resources, trai

{'id': 80, 'job_title': 'Junior MES Engineer| Information Systems', 'location': 'Myrtle Beach, South Carolina Area', 'similarity_score': 0.75, 'explanation': 'The term "Software Engineer" shares significant semantic overlap with "Information Systems" due to the common domain of software development and IT infrastructure. However, the specific inclusion of "MES" (Manufacturing Execution System) in the candidate\'s title indicates a specialized focus on industrial automation and manufacturing software, which is a narrower domain than general software engineering. Consequently, while they are closely related, they are not a perfect match.'}

{'id': 34, 'job_title': 'People Development Coordinator at Ryan', 'location': 'Denton, Texas', 'similarity_score': 0.2, 'explanation': 'The search term "Software Engineer" requires technical proficiency in coding, software development, and computer science. The candidate\'s title, "People Development Coordinator," focuses on human resources, training, and organizational development. These are distinct roles with little to no overlap in skills or responsibilities.'}

{'id': 52, 'job_title': 'Student at Humber College and Aspiring Human Resources Generalist', 'location': 'Kanada', 'similarity_score': 0.2, 'explanation': "There is very low similarity. The search term is a specific technical role ('Software Engineer'), whereas the candidate's title is an academic status ('Student') and a different, non-technical role ('Human Resources Generalist'). They belong to completely different career fields."}]


In [ ]:
print(rank_profile_llm_obj.return_top_profiles("AI", 3))

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Processing 104 candidates for search term 'AI'...
0


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


1


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


...

101


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


102


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


103
[{'id': 12, 'job_title': 'SVP, CHRO, Marketing & Communications, CSR Officer | ENGIE | Houston | The Woodlands | Energy | GPHR | SPHR', 'location': 'Houston, Texas Area', 'similarity_score': 0.85, 'explanation': "The candidate's title is a perfect match for the 'AI' search term in terms of industry, as 'Energy' is a primary sector for AI applications (e.g., AI in energy management). Additionally, 'SPHR' (Senior Professional in Human Resources) and 'CHRO' (Chief Human Resources Officer) are highly relevant to AI, as HR professionals are increasingly involved in AI implementation, talent management, and ethical considerations. The title"}, {'id': 53, 'job_title': 'Seeking Human Resources HRIS and Generalist Positions', 'location': 'Greater Philadelphia Area', 'similarity_score': 0.85, 'explanation': 'The candidate\'s job title is a near-perfect match for the search term. The title specifically requests an HRIS (Human Resources Information System) role, which is the primary technical 

{'id': 12, 'job_title': 'SVP, CHRO, Marketing & Communications, CSR Officer | ENGIE | Houston | The Woodlands | Energy | GPHR | SPHR', 'location': 'Houston, Texas Area', 'similarity_score': 0.85, 'explanation': "The candidate's title is a perfect match for the 'AI' search term in terms of industry, as 'Energy' is a primary sector for AI applications (e.g., AI in energy management). Additionally, 'SPHR' (Senior Professional in Human Resources) and 'CHRO' (Chief Human Resources Officer) are highly relevant to AI, as HR professionals are increasingly involved in AI implementation, talent management, and ethical considerations. The title"}

{'id': 53, 'job_title': 'Seeking Human Resources HRIS and Generalist Positions', 'location': 'Greater Philadelphia Area', 'similarity_score': 0.85, 'explanation': 'The candidate\'s job title is a near-perfect match for the search term. The title specifically requests an HRIS (Human Resources Information System) role, which is the primary technical focus of the AI search term in this context. Additionally, the inclusion of "Generalist" confirms that the candidate is willing to handle the broader HR responsibilities that typically accompany an HRIS role.'}

{'id': 62, 'job_title': 'Seeking Human Resources HRIS and Generalist Positions', 'location': 'Greater Philadelphia Area', 'similarity_score': 0.85, 'explanation': "The search term 'AI' (Artificial Intelligence) is highly semantically related to 'HRIS' (Human Resources Information System). AI is a core technology driving modern HRIS platforms, enabling capabilities like AI-driven recruitment, chatbots for employee support, and analytics for workforce planning. While 'Generalist' refers to the functional role, the specific mention of 'HRIS' strongly indicates a role heavily focused on technology, data, and automation"}]

## Gemma4 model

### Collective Approach

In this approach, I delegate the entire ranking task to the LLM. I use listwise context by feeding the search term and the entire corpus of the job profiles to the context window.

#### Ollama

In [ ]:
!ollama pull gemma4:31b

In [ ]:
rankerOllama_gemma = RankCandidatesLLMCollectiveOllama(model_name="gemma4:31b")

In [ ]:
print(rankerOllama_gemma.rank_candidates(data, 'Business', top_n=5))

Delegating collective ranking to Ollama via LangChain for: 'Business'...
Based on the search term **"Business,"** I have ranked the candidates by prioritizing those with explicit "Business" keywords in their degree or job title, as well as those in high-level executive leadership roles that oversee business operations.

### Top 5 Candidates for Search Term: 'Business'

| Candidate ID | Job Title | Similarity Score | Explanation |
| :--- | :--- | :--- | :--- |
| **ID 81** | Senior Human Resources Business Partner at Heil Environmental | **0.98** | This is the strongest match. The "Business Partner" role is specifically designed to align HR strategy with business objectives. |
| **ID 12** | SVP, CHRO, Marketing & Communications, CSR Officer | **0.95** | Holds a Senior Vice President (SVP) level position, indicating top-tier business leadership and multi-departmental oversight. |
| **ID 102** | Business Intelligence and Analytics at Travelers | **0.95** | Direct keyword match. Business In

## GLM

### Collective Approach

#### Ollama

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,341 kB]
Hit:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
!ollama pull glm-4.7-flash

In [ ]:
!pip install langchain-ollama

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from IPython.display import Markdown

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

model = OllamaLLM(model="glm-4.7-flash")

chain = prompt | model

display(Markdown(chain.invoke({"question": "What are colors similar to ruby red"})))

Here is the step-by-step breakdown:

1.  **Identify the base hue:** Ruby red is a deep, rich, and vibrant shade of red, often associated with the transparency and luminosity of the gemstone.
2.  **Identify variations in tone:** Depending on the light, it can have subtle pink or purple undertones, but it is fundamentally a dark red.
3.  **List similar colors:**
    *   **Crimson:** A deep, bright red that is very similar in intensity.
    *   **Burgundy:** A dark, wine-like red.
    *   **Scarlet:** A bright, intense red, though often slightly more orange than ruby.
    *   **Maroon:** A darker, brownish-red.
    *   **Garnet:** The deep red color of the other famous gemstone.
    *   **Blood Red:** An intense, almost blackish red.

Answer: Some colors similar to ruby red include **crimson**, **burgundy**, **scarlet**, **garnet**, and **maroon**.

In [ ]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
import duckdb

class RankCandidatesLLMCollectiveOllama:
    def __init__(self, model_name="glm-4.7-flash"):
        # Initialize the LangChain Ollama model wrapper
        self.llm = OllamaLLM(model=model_name)

    def rank_candidates(self, duckdb_relation, search_term, top_n=5):
        # Fetch data from DuckDB
        candidates_df = duckdb_relation.fetchdf()

        # Prepare the list of candidates for the prompt
        candidates_text = "\n".join([
            f"ID {row['id']}: {row['job_title']}"
            for _, row in candidates_df.iterrows()
        ])

        # Define the prompt template
        template = """As an expert recruiter, rank the following candidates based on the search term: '{search_term}'.

Candidates:
{candidates_text}

Identify the top {top_n} candidates. For each, provide:
1. The Candidate ID
2. The Candidate's job title
3. A Similarity Score (0.0 to 1.0)
4. A brief explanation of why they are a good match.

Please format your response as a clear list or table."""

        prompt = ChatPromptTemplate.from_template(template)

        # Create the chain using the pipe operator
        chain = prompt | self.llm

        print(f"Delegating collective ranking to Ollama via LangChain for: '{search_term}'...")

        # Invoke the chain
        response = chain.invoke({
            "search_term": search_term,
            "candidates_text": candidates_text,
            "top_n": top_n
        })

        return response

In [ ]:
rankerOllama = RankCandidatesLLMCollectiveOllama()

In [ ]:
import requests
try:
    response = requests.get('http://localhost:11434/api/tags')
    if response.status_code == 200:
        print('Ollama server is running.')
        # If running, proceed with ranking
        ranking_report = rankerOllama.rank_candidates(data, 'Software Engineer', top_n=5)
        print(ranking_report)
except requests.exceptions.ConnectionError:
    print('Ollama server is NOT running. Please re-run the cell that starts the server (T-U95__dNzS4) and wait a few seconds.')

Ollama server is running.
Delegating collective ranking to Ollama via LangChain for: 'Software Engineer'...
Based on the provided list, there is a significant mismatch between the search query ("Software Engineer") and the candidate profiles, which are predominantly in Human Resources. However, the following candidates possess the strongest technical backgrounds, engineering titles, or industry relevance to software.

### Top Matches for "Software Engineer"

| Rank | Candidate ID | Title | Match Score | Reasoning |
| :--- | :--- | :--- | :--- | :--- |
| **1** | **80** | Junior MES Engineer | Information Systems | **0.90** - Contains the exact keyword "Engineer" and "Information Systems," indicating direct technical involvement in systems. |
| **2** | **86** | Information Systems Specialist | and Programmer | **0.85** - Contains the high-value keyword "Programmer" and an "Information Systems Specialist" role. |
| **3** | **102** | Business Intelligence | and Analytics | **0.70** - While

In [ ]:
import requests
try:
    response = requests.get('http://localhost:11434/api/tags')
    if response.status_code == 200:
        print('Ollama server is running.')
        # If running, proceed with ranking
        ranking_report = rankerOllama.rank_candidates(data, 'Human Resources', top_n=5)
        print(ranking_report)
except requests.exceptions.ConnectionError:
    print('Ollama server is NOT running. Please re-run the cell that starts the server (T-U95__dNzS4) and wait a few seconds.')

Ollama server is running.
Delegating collective ranking to Ollama via LangChain for: 'Human Resources'...
Based on the provided list, here are the top 5 candidates for a search related to "Human Resources," selected for their high seniority, direct role titles, and industry reputation.

### 1. ID 12
*   **Score:** 1.0
*   **Profile:** SVP, CHRO at ENGIE (Energy Utility)
*   **Why they are Top Tier:** This is the highest level of HR leadership in the list. The title "Chief Human Resources Officer" is the exact executive equivalent of "Head of Human Resources." This candidate possesses the highest level of strategic HR experience, organizational design, and C-suite executive presence.

### 2. ID 69
*   **Score:** 1.0
*   **Profile:** Director of Human Resources North America
*   **Why they are Top Tier:** This candidate holds a Director-level title, indicating significant seniority and responsibility for HR operations within a major market (North America). "Director" is a key strategic r

In [ ]:
print(rankerOllama.rank_candidates(data, 'Business', top_n=5))

Delegating collective ranking to Ollama via LangChain for: 'Business'...
Based on the search term "Business," the top 5 candidates have been ranked based on their direct use of the keyword, high-level business titles, or strategic relevance to business operations.

**1. ID 102**
*   **Job Title:** Business Intelligence and Analytics at Travelers
*   **Similarity Score:** 1.0
*   **Explanation:** This candidate is the strongest match as their profile explicitly includes the exact search term "Business Intelligence." This indicates they are directly involved in analytical business functions within a major corporation.

**2. ID 12**
*   **Job Title:** SVP, CHRO, Marketing & Communications, CSR Officer | ENGIE
*   **Similarity Score:** 1.0
*   **Explanation:** This candidate holds an "SVP" (Senior Vice President) title, the highest level of operational management. As the Chief Human Resources Officer (CHRO), they are a key strategic leader in business operations, making them an ideal fit f

# Fine Tuning

In this section, I demonstrate how fine-tuning an LLM on an extended dataset of candidates improves ranking performance. To achieve this efficiently, I utilized Parameter-Efficient Fine-Tuning (PEFT) techniques, specifically QLoRA (Quantized Low-Rank Adaptation).

Instead of retraining all model parameters, QLoRA allows for high-performance adaptation by updating only a small subset of task-specific weights while keeping the base model quantized. This approach optimizes training efficiency and reduces memory requirements without compromising the model's reasoning capabilities.

### What I learned

I have noticed the outputs varied between TPU and GPU environments due to differences in hardware and library configurations. For instance, TPUs often require specific libraries like `torch-xla` for proper operation.

During the fine-tuning process, GPU RAM was frequently reaching capacity. To mitigate this and ensure progress was preserved, I saved the fine tune model externally.

To handle datasets that exceeded the model's context window, I implemented a batch processing strategy.

## Phi2

I explored utilizing Phi-3.5 to leverage its expanded context window; however, this transition introduced technical overhead due to dependency conflicts and library version mismatches within the current environment.

In [ ]:
!pip install -q -U bitsandbytes transformers peft accelerate datasets scipy einops evaluate trl rouge_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 24.8 MB/s eta 0:00:00


In [ ]:
import os

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    Trainer,
    GenerationConfig
)
from tqdm import tqdm
from trl import SFTTrainer
import torch
import time
import pandas as pd
import numpy as np

In [ ]:
from huggingface_hub import interpreter_login

interpreter_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

Enter your token (input will not be visible): ··········
Add token as git credential? [y/N]: N


In [ ]:
!pip install -U openpyxl

In [ ]:
import pandas as pd

fine_tuning_dataset_path = '/content/drive/My Drive/Apziva/Project 3 - Potential Talents/Extended Dataset for Potential Talents.xlsx'
fine_tuning_df = pd.read_excel(fine_tuning_dataset_path)

print("Fine-tuning dataset loaded successfully:")
print(fine_tuning_df.head())

Fine-tuning dataset loaded successfully:
   id                                              title       location  \
0   1  innovative and driven professional seeking a r...  United States   
1   2  ms applied data science student usc research a...  United States   
2   3  computer science student seeking full-time sof...  United States   
3   4  microsoft certified power bi data analyst mba ...  United States   
4   5  graduate research assistant at uab masters in ...  United States   

   screening_score  
0              100  
1              100  
2              100  
3              100  
4              100  


In [ ]:
extended_df = fine_tuning_df

In [ ]:
duckdb_conn.register('fine_tuning_data_duckdb', extended_df)

fine_tuning_data = duckdb_conn.sql("SELECT * FROM fine_tuning_data_duckdb").fetchdf()

print("Extended dataset registered as DuckDB view 'fine_tuning_data_duckdb'.")
print("First 5 rows from DuckDB view:")
duckdb_conn.sql("SELECT * FROM fine_tuning_data_duckdb LIMIT 5").show()

Extended dataset registered as DuckDB view 'fine_tuning_data_duckdb'.
First 5 rows from DuckDB view:
┌───────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────┬─────────────────┐
│  id   │                                                                                                   title                                                                                                   │   location    │ screening_score │
│ int64 │                                                                                                  varchar                                                                                                  │    varchar    │      int64      │
├───────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
extended_data= fine_tuning_data

In [ ]:
compute_dtype = getattr(torch, "float16")
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=False,
    )

I have configured the `BitsAndBytesConfig` to implement 4-bit quantization. This configuration is later integrated with the LoRA (Low-Rank Adaptation) setup to perform **QLoRA** fine-tuning.

In [ ]:
model_name='microsoft/phi-2'
device_map = {"": 0}
original_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                      device_map=device_map,
                                                      quantization_config=bnb_config,
                                                      trust_remote_code=True)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code=True,padding_side="left",add_eos_token=True,add_bos_token=True,use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
eval_tokenizer = AutoTokenizer.from_pretrained(model_name, add_bos_token=True, trust_remote_code=True, use_fast=False)
eval_tokenizer.pad_token = eval_tokenizer.eos_token
def gen(model,p, maxlen=100, sample=True):
  toks = eval_tokenizer(p, return_tensors="pt")
  res = model.generate(**toks.to("cuda"), max_new_tokens=maxlen, do_sample=sample,num_return_sequences=1,temperature=0.1,num_beams=1,top_p=0.95,).to('cpu')
  return eval_tokenizer.batch_decode(res,skip_special_tokens=True)

### Inference without Fine tuning

In [ ]:
%%time
import re
from transformers import set_seed
set_seed(seed)

search_term = "data analyst"
top_n = 3
batch_size = 10

def extract_top_ids(output):
    # Helper to find IDs in the model's response
    return re.findall(r'ID\s*(\d+)', output)

all_winners = []

print(f"Processing {len(fine_tuning_data)} candidates in batches of {batch_size}...")

# Step 1: Batch processing
for i in range(0, len(fine_tuning_data), batch_size):
    batch = fine_tuning_data.iloc[i : i + batch_size]
    candidates_text = "\n".join([f"ID {row['id']}: {row['title']}" for _, row in batch.iterrows()])

    prompt = f"""As an expert recruiter, rank the following candidates based on the search term: '{search_term}'.

Candidates:
{candidates_text}

Identify the top {top_n} candidates. For each, provide the ID and a brief explanation."""

    res = gen(original_model, prompt, maxlen=300)
    all_winners.append(res[0])
    if (i // batch_size) % 5 == 0:
        print(f"Processed {i + len(batch)} candidates...")

# Step 2: Final Ranking
final_prompt = f"""Below are the winners from various recruitment rounds for the term '{search_term}':\n\n"""
final_prompt += "\n".join(all_winners[:20]) # Taking a subset of winners to fit final context
final_prompt += f"\n\nBased on these results, who are the absolute top {top_n} candidates? Format as a table."

print("Generating final ranking...")
final_res = gen(original_model, final_prompt, maxlen=500)

print('-' * 100)
print(f"FINAL RANKING:\n{final_res[0]}")

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Processing 1285 candidates in batches of 10...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Processed 10 candidates...


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

Processed 1260 candidates...


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (9082 > 2048). Running this sequence through the model will result in indexing errors
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generating final ranking...


[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


----------------------------------------------------------------------------------------------------
FINAL RANKING:
Below are the winners from various recruitment rounds for the term 'data analyst':

As an expert recruiter, rank the following candidates based on the search term: 'data analyst'.

Candidates:
ID 1: innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.
ID 2: ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories former data science intern quadratyx actively seeking full time roles in summer 2025
ID 3: computer science student seeking full-time software engineerdeveloper positions ai sql data visualization toolspython ssrs
ID 4: microsoft certified power bi data analyst mba business analytics unt business intelligence engineer data scientist data engineer business analytics predictive analytics statistical analysis ex-ericsson
ID 5: graduate research

In [ ]:
search_term = "data analyst"
top_n = 5

# Prepare the list of all candidates for the prompt
candidates_text = "\n".join([f"ID {row['id']}: {row['title']}" for _, row in fine_tuning_data.iterrows()])

# Construct the prompt for the LLM to rank all candidates at once
prompt = f"""As an expert recruiter, rank the following candidates based on the search term: '{search_term}'.\n\nCandidates:\n{candidates_text}\n\nIdentify the top {top_n} candidates. For each, provide the Candidate ID, the Candidate's job title, a Similarity Score (0.0 to 1.0), and a brief explanation of why they are a good match. Please format your response as a clear list or table."""

print(f"Processing all {len(fine_tuning_data)} candidates at once for search term '{search_term}'...")

# Generate the ranking using the model
# Adjust maxlen based on the expected length of the full ranking output
full_ranking_res = gen(original_model, prompt, maxlen=2000)

print('-' * 100)
print(f"Full Ranking for '{search_term}':\n{full_ranking_res[0]}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (23775 > 2048). Running this sequence through the model will result in indexing errors
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in PhiDecoderLayer. Setting `past_key_values=None`.


Processing all 1285 candidates at once for search term 'data analyst'...
----------------------------------------------------------------------------------------------------
Full Ranking for 'data analyst':
As an expert recruiter, rank the following candidates based on the search term: 'data analyst'.

Candidates:
ID 1: innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.
ID 2: ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories former data science intern quadratyx actively seeking full time roles in summer 2025
ID 3: computer science student seeking full-time software engineerdeveloper positions ai sql data visualization toolspython ssrs
ID 4: microsoft certified power bi data analyst mba business analytics unt business intelligence engineer data scientist data engineer business analytics predictive analytics statistical analysis ex-ericsson
ID 5: graduate r

### Training the Fine Tuned model

In [ ]:
from sklearn.model_selection import train_test_split

# Splitting the data: 80% train, 20% temporary (to be split into val/test)
train_df, temp_df = train_test_split(fine_tuning_df, test_size=0.2, random_state=seed)

# Splitting the temporary 20% into 50% validation and 50% test (10% of total each)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=seed)

print(f"Data splitting complete:")
print(f"- Training set: {len(train_df)} rows")
print(f"- Validation set: {len(val_df)} rows")
print(f"- Test set: {len(test_df)} rows")

# Display a sample of the training data
train_df.head()

Data splitting complete:
- Training set: 1028 rows
- Validation set: 128 rows
- Test set: 129 rows


,id,title,location,screening_score
567,568,data science masters graduate from heriot-watt...,United Kingdom,90
464,465,data science mathematical statistics data anal...,United States,95
183,184,multi-competent generalist,United States,75
1221,1222,DATA ANALYST,India,0
1161,1162,data analyst sql python data visualization,United States,25


In [ ]:
from datasets import Dataset, DatasetDict

# Convert DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Combine into a DatasetDict
raw_datasets = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

def tokenize_function(examples):
    # Create the prompt for the model
    # We are training the model to understand the relationship between job titles and scores
    prompts = [
        f"Job Title: {title}\nLocation: {location}\nScreening Score: {score}"
        for title, location, score in zip(examples['title'], examples['location'], examples['screening_score'])
    ]
    return tokenizer(prompts, truncation=True, padding="max_length", max_length=256)

tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)

print("Tokenization complete. Dataset structure:")
print(tokenized_datasets)

Map:   0%|          | 0/1028 [00:00<?, ? examples/s]

Map:   0%|          | 0/128 [00:00<?, ? examples/s]

Map:   0%|          | 0/129 [00:00<?, ? examples/s]

Tokenization complete. Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1028
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 128
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 129
    })
})


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
original_model = prepare_model_for_kbit_training(original_model)

In [ ]:
config = LoraConfig(
    r=32, #Rank
    lora_alpha=32,
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'dense'
    ],
    bias="none",
    lora_dropout=0.05,  # Conventional
    task_type="CAUSAL_LM",
)

# 1 - Enabling gradient checkpointing to reduce memory usage during fine-tuning
original_model.gradient_checkpointing_enable()

peft_model = get_peft_model(original_model, config)

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

In [ ]:
print(print_number_of_trainable_model_parameters(peft_model))

trainable model parameters: 20971520
all model parameters: 1542364160
percentage of trainable model parameters: 1.36%


In [ ]:
# Define the output directory on Google Drive for persistent storage
output_dir = '/content/drive/My Drive/Apziva/Project 3 - Potential Talents/phi2-finetuned-checkpoints'

In [ ]:
import transformers

# Further optimized for TPU memory constraints
peft_training_args = TrainingArguments(
    output_dir = output_dir,
    warmup_steps=1,
    per_device_train_batch_size=1, # Minimum batch size
    gradient_accumulation_steps=8, # Increased to reduce memory pressure
    max_steps=500, # Reduced steps for initial test
    learning_rate=2e-4,
    optim="adamw_torch",
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    gradient_checkpointing=True, # Essential for saving memory
    report_to="none",
    ddp_find_unused_parameters=False,
    dataloader_drop_last=True
)

peft_model.config.use_cache = False

In [ ]:
peft_trainer = transformers.Trainer(
    model=peft_model,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    args=peft_training_args,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

peft_trainer.train()

Step,Training Loss
10,3.991253
20,2.656355
30,2.372103
40,2.209093
50,2.085833
60,2.000033
70,1.974134
80,2.046409
90,1.953696
100,2.158591


TrainOutput(global_step=500, training_loss=1.9155545024871825, metrics={'train_runtime': 2484.0559, 'train_samples_per_second': 1.61, 'train_steps_per_second': 0.201, 'total_flos': 1.635271440334848e+16, 'train_loss': 1.9155545024871825, 'epoch': 3.8793774319066148})

### Inference with the Fine Tuned model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model_id = "microsoft/phi-2"
base_model = AutoModelForCausalLM.from_pretrained(base_model_id,
                                                      device_map='auto',
                                                      quantization_config=bnb_config,
                                                      trust_remote_code=True)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
eval_tokenizer = AutoTokenizer.from_pretrained(base_model_id, add_bos_token=True, trust_remote_code=True, use_fast=False)
eval_tokenizer.pad_token = eval_tokenizer.eos_token

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def gen(model,p, maxlen=100, sample=True):
  toks = eval_tokenizer(p, return_tensors="pt")
  res = model.generate(**toks.to("cuda"), max_new_tokens=maxlen, do_sample=sample,num_return_sequences=1,temperature=0.1,num_beams=1,top_p=0.95,).to('cpu')
  return eval_tokenizer.batch_decode(res,skip_special_tokens=True)

In [ ]:
from peft import PeftModel
import torch

# Using the local output directory and the latest checkpoint found in your files
checkpoint_path = f"{output_dir}/checkpoint-500"

ft_model_lora = PeftModel.from_pretrained(
    base_model,
    checkpoint_path,
    torch_dtype=torch.float16,
    is_trainable=False
)

In [ ]:
search_term = "data analyst"
top_n = 5

# Prepare the list of all candidates for the prompt
candidates_text = "\n".join([f"ID {row['id']}: {row['title']}" for _, row in fine_tuning_data.iterrows()])

# Construct the prompt for the LLM to rank all candidates at once
prompt = f"""As an expert recruiter, rank the following candidates based on the search term: '{search_term}'.\n\nCandidates:\n{candidates_text}\n\nIdentify the top {top_n} candidates. For each, provide the Candidate ID, the Candidate's job title, a Similarity Score (0.0 to 1.0), and a brief explanation of why they are a good match. Please format your response as a clear list or table."""

print(f"Processing all {len(fine_tuning_data)} candidates at once for search term '{search_term}'...")

# Generate the ranking using the model
# Adjust maxlen based on the expected length of the full ranking output
full_ranking_res_fine_tuned_lora = gen(ft_model_lora, prompt, maxlen=2000)

print('-' * 100)
print(f"Full Ranking for '{search_term}':\n{full_ranking_res_fine_tuned_lora[0]}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (23775 > 2048). Running this sequence through the model will result in indexing errors
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Processing all 1285 candidates at once for search term 'data analyst'...


[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


----------------------------------------------------------------------------------------------------
Full Ranking for 'data analyst':
As an expert recruiter, rank the following candidates based on the search term: 'data analyst'.

Candidates:
ID 1: innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.
ID 2: ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories former data science intern quadratyx actively seeking full time roles in summer 2025
ID 3: computer science student seeking full-time software engineerdeveloper positions ai sql data visualization toolspython ssrs
ID 4: microsoft certified power bi data analyst mba business analytics unt business intelligence engineer data scientist data engineer business analytics predictive analytics statistical analysis ex-ericsson
ID 5: graduate research assistant at uab masters in data science student at uab ex jio
ID

In [ ]:
import re

def extract_ranked_ids(output):
    # Helper to find IDs mentioned in the model's response
    return re.findall(r'ID\s*(\d+)', output)

search_term = "data analyst"
batch_size = 10
top_per_batch = 3
batch_winners = []

# Filter out IDs with 'None' titles before processing
valid_candidates = fine_tuning_data[fine_tuning_data['title'].notnull()]

print(f"Processing {len(valid_candidates)} candidates in batches of {batch_size}...")

# Step 1: Batch Processing
for i in range(0, len(valid_candidates), batch_size):
    batch = valid_candidates.iloc[i : i + batch_size]
    candidates_text = "\n".join([f"ID {row['id']}: {row['title']}" for _, row in batch.iterrows()])

    prompt = f"""As an expert recruiter, rank the following candidates based on the search term: '{search_term}'.

Candidates:
{candidates_text}

Identify the top {top_per_batch} candidates. For each, provide the ID and a brief explanation."""

    # Using the fine-tuned model (ft_model_lora) from previous cells
    res = gen(ft_model_lora, prompt, maxlen=5000)
    batch_winners.append(res[0])

    if (i // batch_size) % 10 == 0:
        print(f"Processed {i + len(batch)} candidates...")


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Processing 1281 candidates in batches of 10...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Processed 10 candidates...


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Processed 1110 candidates...


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Processed 1210 candidates...


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [ ]:
print(batch_winners)

["As an expert recruiter, rank the following candidates based on the search term: 'data analyst'.\n\nCandidates:\nID 1: innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.\nID 2: ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories former data science intern quadratyx actively seeking full time roles in summer 2025\nID 3: computer science student seeking full-time software engineerdeveloper positions ai sql data visualization toolspython ssrs\nID 4: microsoft certified power bi data analyst mba business analytics unt business intelligence engineer data scientist data engineer business analytics predictive analytics statistical analysis ex-ericsson\nID 5: graduate research assistant at uab masters in data science student at uab ex jio\nID 6: student at kennesaw state university\nID 7: data analyst business analyst python snowflake sql machine learning power bi

In [ ]:
print(batch_winners)

["As an expert recruiter, rank the following candidates based on the search term: 'data analyst'.\n\nCandidates:\nID 1: innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.\nID 2: ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories former data science intern quadratyx actively seeking full time roles in summer 2025\nID 3: computer science student seeking full-time software engineerdeveloper positions ai sql data visualization toolspython ssrs\nID 4: microsoft certified power bi data analyst mba business analytics unt business intelligence engineer data scientist data engineer business analytics predictive analytics statistical analysis ex-ericsson\nID 5: graduate research assistant at uab masters in data science student at uab ex jio\nID 6: student at kennesaw state university\nID 7: data analyst business analyst python snowflake sql machine learning power bi

In [ ]:
search_term = "data analyst"

In [ ]:
import re
import torch
import gc

# Clear CUDA cache and reset peak memory stats
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
gc.collect()

def extract_ranked_from_batch_winner(batch_winner_text):
    candidates = []
    answer_section_match = re.search(r"Answer:\n(.*?)(?=\nExplanation:|$)", batch_winner_text, re.DOTALL)
    if not answer_section_match:
        return []
    answer_content = answer_section_match.group(1).strip()
    ranked_lines = re.findall(r'^\s*\d+\.\s*(ID\s*\d+:\s*.*?)(?=\n\d+\. ID|$)', answer_content, re.MULTILINE | re.DOTALL)
    for line in ranked_lines:
        match = re.match(r'ID\s*(\d+):\s*(.*)', line.strip())
        if match:
            candidates.append({'id': int(match.group(1)), 'job_title': match.group(2).strip()})
    return candidates

# Stage 2 Distillation: Rename and process batch results
semi_final_batch_size = 10
all_top_candidates_from_batches = []
for batch_output in batch_winners:
    all_top_candidates_from_batches.extend(extract_ranked_from_batch_winner(batch_output))

concise_candidate_list_for_distillation = []
seen_ids = set()
for candidate in all_top_candidates_from_batches:
    if candidate['id'] not in seen_ids:
        concise_candidate_list_for_distillation.append(f"Candidate ID: {candidate['id']}, Job Title: {candidate['job_title']}")
        seen_ids.add(candidate['id'])

semi_final_winners = []
for i in range(0, len(concise_candidate_list_for_distillation), semi_final_batch_size):
    batch = concise_candidate_list_for_distillation[i : i + semi_final_batch_size]
    batch_text = "\n".join(batch)
    distill_prompt = f"""From the following candidates for '{search_term}', pick ONLY the top 2 most relevant. Output only ID and Title.\n\nCandidates:\n{batch_text}"""
    distilled_res = gen(ft_model_lora, distill_prompt, maxlen=150)
    semi_final_winners.append(distilled_res[0])

# Explicit memory cleanup before final ranking
torch.cuda.empty_cache()
gc.collect()

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `e

57

In [ ]:
semi_final_winners

["From the following candidates for 'data analyst', pick ONLY the top 2 most relevant. Output only ID and Title.\n\nCandidates:\nCandidate ID: 1, Job Title: innovative and driven professional seeking a role in data analytics\nCandidate ID: 4, Job Title: microsoft certified power bi data analyst\nCandidate ID: 2, Job Title: ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories\nCandidate ID: 11, Job Title: Experienced Data Analyst\nCandidate ID: 14, Job Title: Collaborative Data Engineer\nCandidate ID: 19, Job Title: Senior Technology Analyst\nCandidate ID: 42, Job Title: Python SQL Data Integration Pyspark Machine Learning LLM Apache Airflow ETL Meng Graduate\nCandidate ID: 52, Job Title: data analyst ai engineer bi engineer\nCandidate ID: 64, Job Title: M.S. Computer Science Graduate - Machine Learning Data Science Enthusiast\nCandidate ID: 73, Job Title: Program Analyst at Cognizant - This candidate has a strong background i

In [ ]:
# Final consolidation using renamed distilled results
final_prompt_candidates_string = "\n".join(semi_final_winners[:10])
final_prompt = f"""Rank the top 5 candidates for '{search_term}' from this list:\n{final_prompt_candidates_string}\n\nFormat as Markdown table: | Rank | ID | Job Title | Score (0-1) |\nFollowed by '### Justification' bullet points."""

print(f"\nGenerating final absolute top 5 ranking (Distillation batch: {semi_final_batch_size})...")
final_top_5 = gen(ft_model_lora, final_prompt, maxlen=600)

print('-' * 100)
print(f"FINAL TOP 5 RANKING:\n{final_top_5[0]}")

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Generating final absolute top 5 ranking (Distillation batch: 10)...
----------------------------------------------------------------------------------------------------
FINAL TOP 5 RANKING:
Rank the top 5 candidates for 'data analyst' from this list:
From the following candidates for 'data analyst', pick ONLY the top 2 most relevant. Output only ID and Title.

Candidates:
Candidate ID: 1, Job Title: innovative and driven professional seeking a role in data analytics
Candidate ID: 4, Job Title: microsoft certified power bi data analyst
Candidate ID: 2, Job Title: ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories
Candidate ID: 11, Job Title: Experienced Data Analyst
Candidate ID: 14, Job Title: Collaborative Data Engineer
Candidate ID: 19, Job Title: Senior Technology Analyst
Candidate ID: 42, Job Title: Python SQL Data Integration Pyspark Machine Learning LLM Apache Airflow ETL Meng Graduate
Candidate ID: 52, Job Title: da

# RAG (Retrieval-Augmented Generation)

In this section, I implemented a RAG pipeline to extend the ranking system's capabilities beyond the original candidate dataset.

By leveraging a vector-based retrieval mechanism (FAISS), the LLM can dynamically access and evaluate candidates from the extended dataset that were not part of the original corpus, ensuring a more comprehensive ranking process.

In [ ]:
%pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 23.4 MB/s eta 0:00:00


In [ ]:
import faiss
import torch
from transformers import AutoTokenizer, AutoModel
from transformers import AutoModelForSeq2SeqLM

In [ ]:
# Model to use in retriever
retriever_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
retriever_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def generate_embedding(docs, model, tokenizer):
    # Tokenize each text and convert to PyTorch tensors
    inputs = tokenizer(docs, padding=True, truncation=True, return_tensors="pt", max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)

    # Embedding defined as mean pooling of all tokens
    attention_mask = inputs["attention_mask"]
    embeddings = outputs.last_hidden_state

    expanded_mask = attention_mask.unsqueeze(-1).expand(embeddings.shape).float()
    sum_embeddings = torch.sum(embeddings * expanded_mask, axis=1)
    sum_mask = torch.clamp(expanded_mask.sum(axis=1), min=1e-9)
    mean_embeddings = sum_embeddings / sum_mask

    # Convert to numpy array
    return mean_embeddings.cpu().numpy()

def retrieve_documents(query, index, documents, k=3):
    # Generate embedding for the query
    query_embedding = generate_embedding(query, retriever_model, retriever_tokenizer)   # 1xD matrix
    # Search the index for similar documents
    distances, indices = index.search(query_embedding, k)  # 1xk matrices
    # Return the retrieved documents and their distances
    retrieved_docs = [(documents[idx], float(distances[0][i])) for i, idx in enumerate(indices[0])]
    return retrieved_docs

def rag_pipeline(query, documents, retriever_k=3, max_length=150):
    retrieved_docs = retrieve_documents(query, index, documents, k=retriever_k)
    docs = [doc for doc, distance in retrieved_docs]
    response = generate_response(query, docs, max_length=max_length)
    return response, retrieved_docs

## GLM

In [ ]:
import ollama

def generate_response(query, retrieved_docs, max_length=150):
    # Combine the retrieved documents into a single text block for the prompt
    if not retrieved_docs:
        return "No relevant candidates were found to rank."

    candidates_text = "\n".join([f"- {doc}" for doc in retrieved_docs])

    prompt = f"""As an expert recruiter, rank the following candidates based on the search term: '{query}'.

Candidates retrieved from database:
{candidates_text}

For each candidate, provide a Similarity Score (0.0 to 1.0) and a brief justification.
Format your response as a clear Markdown list."""

    # Invoke the GLM model via Ollama
    response = ollama.generate(model='glm-4.7-flash', prompt=prompt)

    return response['response']

In [ ]:
import pandas as pd

# Extract the job titles from the extended dataset to serve as the document corpus for retrieval
# We use the 'title' column from fine_tuning_df which contains the rich descriptions
documents = extended_df['title'].astype(str).tolist()

print(f"Converted {len(documents)} entries from the extended dataset into document format.")
print("Sample document:")
print(documents[0])

Converted 1285 entries from the extended dataset into document format.
Sample document:
innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.


In [ ]:
# Generate embeddings for all documents, then create FAISS index for efficient similarity search
document_embeddings = generate_embedding(documents, retriever_model, retriever_tokenizer)
dimension = document_embeddings.shape[1]   # Dimension of the embeddings
index = faiss.IndexFlatL2(dimension)       # Using L2 (Euclidean) distance
index.add(document_embeddings)             # Add embeddings to the index
print(f"Created index with {index.ntotal} documents")

Created index with 1285 documents


In [ ]:
query = "Software Engineer"
response, retrieved_docs = rag_pipeline(query, documents, retriever_k=5)

print(f"Query: {query}")
print("\n--- Retrieved from FAISS index ---")
for i, (doc, distance) in enumerate(retrieved_docs):
    print(f"{i+1}. (L2 Distance: {distance:.4f}) {doc}")

print("\n--- GLM-4 Expert Ranking ---")
from IPython.display import Markdown, display
display(Markdown(response))

Query: Software Engineer

--- Retrieved from FAISS index ---
1. (L2 Distance: 0.0000) Software Engineer
2. (L2 Distance: 0.0000) software engineer
3. (L2 Distance: 0.0000) software engineer
4. (L2 Distance: 0.0000) software engineer
5. (L2 Distance: 0.0000) software engineer

--- GLM-4 Expert Ranking ---


Here are the ranked candidates based on the search term 'Software Engineer':

1.  **Software Engineer**
    *   **Similarity Score:** 1.0
    *   **Justification:** Exact character-for-character match with the search term.

2.  **software engineer**
    *   **Similarity Score:** 1.0
    *   **Justification:** Perfect case-insensitive match; semantically identical to the search term.

3.  **software engineer**
    *   **Similarity Score:** 1.0
    *   **Justification:** Perfect case-insensitive match; semantically identical to the search term.

4.  **software engineer**
    *   **Similarity Score:** 1.0
    *   **Justification:** Perfect case-insensitive match; semantically identical to the search term.

5.  **software engineer**
    *   **Similarity Score:** 1.0
    *   **Justification:** Perfect case-insensitive match; semantically identical to the search term.

In [ ]:
query = "Data Science professional with NLP experience"
response, retrieved_docs = rag_pipeline(query, documents, retriever_k=5)

print(f"Query: {query}")
print("\n--- Retrieved from FAISS index ---")
for i, (doc, distance) in enumerate(retrieved_docs):
    print(f"{i+1}. (L2 Distance: {distance:.4f}) {doc}")

print("\n--- GLM-4 Expert Ranking ---")
from IPython.display import Markdown, display
display(Markdown(response))

Query: Data Science professional with NLP experience

--- Retrieved from FAISS index ---
1. (L2 Distance: 20.6183) aiml enthusiast recent b.tech graduate deep learning nlp data science
2. (L2 Distance: 21.6152) actively seeking full time data analystdata scientist roles python r sql powerbi machine learning nlp pl300 certified
3. (L2 Distance: 22.0310) data science machine learning artificial intelligence nlp
4. (L2 Distance: 22.1520) data engineer scientist phd in computational engineering skilled in python pytorch aws databricks expertise in machine learning nlp scalable data solutions
5. (L2 Distance: 22.4260) open to data-driven opportunities 4 years experience data engineering data analyst data science masters student at university of new haven python sql unix power bi aws

--- GLM-4 Expert Ranking ---


Here is the ranking of the candidates based on the search term **'Data Science professional with NLP experience'**:

1.  **Candidate 4**
    *   **Similarity Score:** 0.95
    *   **Justification:** This candidate explicitly states "expertise in machine learning nlp" and holds a PhD in Computational Engineering, indicating high-level academic and professional capability in the specific domain.

2.  **Candidate 2**
    *   **Similarity Score:** 0.90
    *   **Justification:** Directly targets "data scientist roles" and lists "nlp" alongside a strong stack of tools (Python, SQL) and a specific certification (PL300), indicating a ready-to-hire professional with the required skill.

3.  **Candidate 1**
    *   **Similarity Score:** 0.85
    *   **Justification:** Clearly identifies as an "aiml enthusiast" and lists "nlp" and "deep learning" as key skills, though the "recent b.tech graduate" status suggests less seniority than the top-ranked candidates.

4.  **Candidate 3**
    *   **Similarity Score:** 0.70
    *   **Justification:** Contains the core keywords ("data science", "nlp") but lacks context regarding experience level or specific technical depth.

5.  **Candidate 5**
    *   **Similarity Score:** 0.15
    *   **Justification:** The candidate possesses strong Data Science experience (4 years, Masters student) but the profile **does not contain the keyword 'NLP'**. Therefore, they are disqualified for this specific search query.

In [ ]:
query = "Machine Learning Engineer specializing in high-energy physics and particle data analysis"
response, retrieved_docs = rag_pipeline(query, documents, retriever_k=5)

print(f"Query: {query}")
print("\n--- Retrieved from FAISS index ---")
for i, (doc, distance) in enumerate(retrieved_docs):
    print(f"{i+1}. (L2 Distance: {distance:.4f}) {doc}")

print("\n--- GLM-4 Expert Ranking ---")
from IPython.display import Markdown, display
display(Markdown(response))

Query: Machine Learning Engineer specializing in high-energy physics and particle data analysis

--- Retrieved from FAISS index ---
1. (L2 Distance: 19.2037) Machine Learning Engineer Google Certified Associate Cloud Engineer M-Tech Silver Medalist in Artificial Intelligence Data Science.
2. (L2 Distance: 19.9358) physics ms analytical problem solver proficient in data analysis and visualization and machine learning
3. (L2 Distance: 19.9387) Machine Learning Engineer at Cedar Gate Technologies Ex- Darazian Daraz Nepal Alibaba Group
4. (L2 Distance: 19.9690) machine learning engineer specializing in ai pipelines mlops and scalable solutions automating workflows to drive impact and learn continuously.
5. (L2 Distance: 20.0190) data scientist ai machine learning engineer graduate assistant statistician drug discovery scientist

--- GLM-4 Expert Ranking ---


Here are the ranked candidates based on their similarity to the search term **"Machine Learning Engineer specializing in high-energy physics and particle data analysis"**.

*   **Candidate:** physics ms analytical problem solver proficient in data analysis and visualization and machine learning
    *   **Similarity Score:** 0.65
    *   **Justification:** This candidate is the top match because it explicitly contains the specific domain keyword "physics," which is the primary focus of the search query. While the title is less specific, the background is the closest to high-energy physics.

*   **Candidate:** machine learning engineer specializing in ai pipelines mlops and scalable solutions automating workflows to drive impact and learn continuously.
    *   **Similarity Score:** 0.55
    *   **Justification:** This candidate holds the exact job title "Machine Learning Engineer" and describes technical skills (MLOps, scalable solutions) that are critical for processing the large datasets typically found in high-energy physics, even though the specific domain keyword is missing.

*   **Candidate:** Machine Learning Engineer Google Certified Associate Cloud Engineer M-Tech Silver Medalist in Artificial Intelligence Data Science.
    *   **Similarity Score:** 0.40
    *   **Justification:** Matches the job title well and demonstrates academic rigor with a silver medalist credential, but lacks any mention of physics or particle data analysis.

*   **Candidate:** Machine Learning Engineer at Cedar Gate Technologies Ex- Darazian Daraz Nepal Alibaba Group
    *   **Similarity Score:** 0.40
    *   **Justification:** Matches the job title well and has relevant industry experience, but the background is in e-commerce rather than high-energy physics.

*   **Candidate:** data scientist ai machine learning engineer graduate assistant statistician drug discovery scientist
    *   **Similarity Score:** 0.20
    *   **Justification:** This candidate focuses on "drug discovery," which is a biological domain, not the physical science of high-energy physics, making them the least relevant match.

## Deepseek

In [ ]:
# 1. Install dependencies, Ollama binary, and python library
!sudo apt-get update && sudo apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama

import ollama
import threading
import subprocess
import time
import requests
from IPython.display import Markdown, display

# 2. Start Ollama service in the background
def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])

# Check if service is already running, if not, start it
try:
    requests.get('http://localhost:11434/api/tags')
    print("Ollama service already running.")
except requests.exceptions.ConnectionError:
    print("Starting Ollama service...")
    serve_thread = threading.Thread(target=run_ollama_serve)
    serve_thread.daemon = True
    serve_thread.start()

# Wait for the service to be responsive
print("Waiting for Ollama service to respond...")
for i in range(15):
    try:
        res = requests.get('http://localhost:11434/api/tags')
        if res.status_code == 200:
            print("Ollama service is up and running!")
            break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
else:
    print("Error: Ollama service failed to start.")

# 3. Pull the DeepSeek model
# Using a standard deepseek-v2 tag
print("Pulling DeepSeek-V2 model...")
!ollama pull deepseek-v2


In [ ]:

# 4. Define the ranking function
def generate_response_deepseek(query, retrieved_docs, max_length=500):
    if not retrieved_docs:
        return "No relevant candidates were found to rank."
    candidates_text = "\n".join([f"- {doc}" for doc in retrieved_docs])
    prompt = f"""As an expert recruiter specializing in quantitative finance, rank the following candidates based on the search term: '{query}'.\n\nCandidates retrieved from database:\n{candidates_text}\n\nFor each candidate, provide a Similarity Score (0.0 to 1.0) and a brief justification focusing on their suitability for a Quant Developer role.\nFormat your response as a clear Markdown list."""
    response = ollama.generate(model='deepseek-v2', prompt=prompt)
    return response['response']

# 5. Run the RAG pipeline
query_quant = "quant developers"
num_candidates = 5 # <--- Specify the number of candidates to showcase here

import __main__
__main__.generate_response = generate_response_deepseek

# Updated call to use the num_candidates variable
response_text, retrieved_docs = rag_pipeline(query_quant, documents, retriever_k=num_candidates)

print(f"Query: {query_quant}")
print(f"\n--- Retrieved top {num_candidates} from FAISS index ---")
for i, (doc, distance) in enumerate(retrieved_docs):
    print(f"{i+1}. (L2 Distance: {distance:.4f}) {doc}")

print("\n--- DeepSeek Expert Ranking ---")
display(Markdown(response_text))

Query: quant developers

--- Retrieved top 5 from FAISS index ---
1. (L2 Distance: 32.5075) CAD Design IOT Developer Fullstack AI Developer Resercher Analyst Animator Building Innovative Products for the Future
2. (L2 Distance: 33.2727) full stack developer m.s. computer science at georgia institute of technology b.s. computer science at university of akron
3. (L2 Distance: 33.5477) software developer
4. (L2 Distance: 33.5477) software developer
5. (L2 Distance: 34.0649) full-stack developer mechanical engineer encoding the future

--- DeepSeek Expert Ranking ---


 | Candidate                     | Similarity Score | Justification                                                                                                       |
|---------------------------------|------------------|-----------------------------------------------------------------------------------------------------------------------|
| CAD Design IoT Developer      | 0.85             | The candidate's focus on design and development of Internet of Things (IoT) solutions, which can include quantitative finance applications such as market monitoring or algorithmic trading systems, is highly relevant to the role of a Quant Developer. |
| Full Stack Developer MS-CS at Georgia Tech & B.S. CS at University of Akron  | 0.8             | The candidate's advanced education and experience in computer science from prestigious institutions like Georgia Institute of Technology positions them well for technical roles requiring deep understanding of both front-end and back-end development processes, which are essential skills for a Quant Developer. |
| Software Developer            | 0.75             | This is a broad term that can encompass various roles within the technology sector including those focused on finance software development. The candidate could potentially be suitable if their experience aligns with developing financial applications or quantitative tools. |
| Full-stack developer (Mechanical Engineer) | 0.65            | Although not directly in quantitative finance, a full-stack developer who has worked on complex systems integration and process automation would have skills that are highly transferable to the development of advanced analytical software used in financial roles like Quant Developer. |
| Software Developer           | 0.7              | Similar reasoning as above but with slightly less specific justification; this candidate could be suitable if their skill set matches what is needed for a quant developer role, particularly within large tech firms where there might be multiple areas of application for technical skills in finance. |

## Gemini

### Note on API Quotas

When I attempted to execute the pipeline using **Gemini 2.0 Flash**, I encountered a `429 Quota Exceeded` error.

I learned that I can monitor real-time API usage and manage project quotas throught the [Google AI Studio Usage Monitor](https://ai.dev/rate-limit).

In [ ]:
import google.generativeai as genai
from google.colab import userdata
from IPython.display import Markdown, display
import __main__

# 1. Setup Gemini API using Colab Secrets
try:
    # Configure the SDK with your API key
    genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

    # Use Gemini 2.0 Flash
    MODEL_ID = 'gemini-2.0-flash'
    client = genai.GenerativeModel(MODEL_ID)
except Exception as e:
    print("Error: Please add your API key to Colab Secrets as 'GEMINI_API_KEY'.")

# 2. Define the Gemini Generator Function
def generate_response_gemini(query, retrieved_docs, max_length=1000):
    if not retrieved_docs:
        return "No relevant candidates were found to rank."

    candidates_text = "\n".join([f"- {doc}" for doc in retrieved_docs])

    prompt = f"""As an expert recruiter specializing in quantitative finance and technology,
rank the following candidates based on the search term: '{query}'.

Candidates retrieved from database:
{candidates_text}

For each candidate, provide:
1. A Similarity Score (0.0 to 1.0)
2. A brief, professional justification for the score.

Format your response as a structured Markdown list or table."""

    # Generate content using the SDK
    response = client.generate_content(prompt)
    return response.text

# 3. Integrate into the RAG Pipeline
__main__.generate_response = generate_response_gemini

# 4. Execute Pipeline
query_tech = "quant developers with high-frequency trading experience"
num_candidates = 5

try:
    response_text, retrieved_docs = rag_pipeline(query_tech, documents, retriever_k=num_candidates)

    print(f"Query: {query_tech}")
    print(f"\n--- FAISS Retrieval: Top {num_candidates} Candidates ---")
    for i, (doc, distance) in enumerate(retrieved_docs):
        print(f"{i+1}. [Dist: {distance:.4f}] {doc}")

    print(f"\n--- {MODEL_ID} Expert Ranking ---")
    display(Markdown(response_text))
except Exception as e:
    print(f"Error: {e}")

Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 49.462177175s.


# Gemini App

I developed a [Talent Match and Ranker app](https://ai.studio/apps/f6cb87d2-8383-449c-a658-3c16191f982e) that ranks candidates based on search terms. Powered by a LangGraph structure, the system intelligently detects and geographical details—automatically triggering a mapping tool to recalculate candidate scores based on location proximity.

# Conclusion


Throughout this project, I explored a spectrum of methodologies for candidate ranking, ranging from statistical approaches to state-of-the-art Large Language Models (LLMs).

1.  **TF-IDF Vectorization**: I began with TF-IDF, which demonstrated how word frequency and uniqueness within a corpus can be used to determine relevance. While effective for keyword matching, it lacked a deeper semantic understanding.
2.  **Word Embeddings**: Using word embeddings (Word2Vec, GloVe, FastText), I mathematically calculated semantic similarities by tokenizing the corpus and mapping each token to its corresponding vector space representation. This stage addressed the 'Out-of-Vocabulary' (OOV) limitation, identifying how unseen words could lead to errors.
3.  **Large Language Models (LLMs)**: Delegating the ranking task to LLMs (like Qwen and GLM) provided a significant improvement in performance and interpretability. The models provided high-quality reasoning for each ranking decision, moving the system closer to human-like evaluation. Using **Ollama** locally allowed for optimized inference.
4.  **Fine-Tuning**: To adapt the models to specific talent acquisition datasets, I utilized **QLoRA** for parameter-efficient fine-tuning. This process improved the model's ability to handle job titles outside the original training set, though it introduced considerations regarding model storage and deployment overhead.
5.  **Retrieval-Augmented Generation (RAG)**: I implemented a RAG pipeline leveraging **FAISS** to dynamically retrieve and inject relevant candidate profiles into the model's context during inference. This architecture provides a highly scalable solution, enabling the system to rank new, unseen data in real-time without the need for additional model training.
6.  **Gemini Integration & Agentic Workflows**: Finally, I explored the **Gemini API** to develop a dedicated **Talent Matcher App**. Despite free-tier rate limits, the app successfully showcased **agentic workflows** using **LangGraph**. The system intelligently detected geographical intent in search terms, automatically calling relevant tools to calculate location-based proximity.